In [ ]:
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)

import prism
import pym

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA, EPG_TECHNOLOGIES_FINAL
from imagematerials.electricity.utils import rebroadcast_type_dims

from imagematerials.model import (
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    EvBatteryLinkModule
)
from imagematerials.factory import ModelFactory
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.concepts import create_electricity_graph, create_materials_graph, KnowledgeGraph, Node
from imagematerials.util import resolve_circular_economy_scenario
from imagematerials.read_mym import read_mym_df, mymarray_to_xarray


# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")
ce_scenario_path = Path(path_base, "circular_economy_scenarios")

COLOURS_HYDROGEN = {
    'ClGs':   '#1B9E77',
    'POOl':   '#D95F02',
    'SMR':    '#7570B3',
    'BioGs':  '#66A61E',
    'ClGsCS': '#66C2A5',
    'POOlCS': '#FC8D62',
    'SMRCS':  '#8DA0CB',
    'BioGsCS':'#A6D854',
    'BLEl':   '#E6AB02',
    'CurEl':  '#E7298A',
    'SPVEl':  '#A6761D',
    'CSPEl':  '#1F78B4',
    'WONEl':  '#B2DF8A',
    'WOFFEl': '#FDBF6F',
    'GEOEl':  '#CAB2D6',
    'ssSMR':  '#6A3D9A',
    'ssEl':   '#B15928',
}

In [ ]:
h2cap = mymarray_to_xarray(pym.read_mym(Path(path_base, "image", "SSP2_baseline", "EnergyServices", "H2Cap.out")),
                          dim_1="NRT", dim_2="HydrogenSourcesTotal")
h2prod = mymarray_to_xarray(pym.read_mym(Path(path_base, "image", "SSP2_baseline", "EnergyServices", "H2ProdSpec.out")),
                          dim_1="NRT", dim_2="HydrogenSources")
h2cap = prism.Q_(h2cap, "GW")
h2prod = prism.Q_(h2prod, "GJ")

h2cap = h2cap.rename({'t': 'time','NRT': 'Region','HydrogenSourcesTotal': 'Type'})
h2prod = h2prod.rename({'t': 'time','NRT': 'Region','HydrogenSources': 'Type'})

h2cap = h2cap.drop_sel(Type='total')


In [ ]:

h2cap

# Plots

In [ ]:
def create_h2_graph():
        """Create the knowledge graph for the vehicle/transport sector (including type synonyms)."""
        h2_knowledge_graph = KnowledgeGraph(Node("HydrogenProduction"))

        tech_types = {
             "ClGs": ["coal gasification"],
             "POOl": ["oil partial oxidation", "partial oxidation of oil"],
            "SMR": ["NG steam reforming", "steam reforming of natural gas"],
            "BioGs": ["biofuel gasification", "gasification of biofuels"],
            "ClGsCS": ["coal gasification + CCS"],
            "POOlCS": ["oil partial oxidation + CCS"],
            "SMRCS": ["NG steam reforming + CCS", "steam reforming of natural gas / autothermal reforming of natural gas + CCS"],
            "BioGsCS": ["biofuel gasification + CCS", "gasification of biofuels + CCS"],
            "BLEl": ["electrolysis - base load", "base-load electrolysis (alkaline/SO)"],
            "CurEl": ["electrolysis - curtailment", "curtailed electricity electrolysis (PEM)"],
            "SPVEl": ["electrolysis + SPV", "electrolyzer (PEM) + solar PV electric power (central)"],
            "CSPEl": ["electrolysis + CSP", "electrolyzer (PEM) + concentrated solar electric power"],
            "WONEl": ["electrolysis - wind (onshore)", "electrolyzer (PEM) + onshore wind electric power"],
            "WOFFEl": ["electrolysis - wind (offshore)", "electrolyzer (PEM) + offshore wind electric power"],
            "GEOEl": ["electrolysis - geothermal", "electrolyzer (PEM) + geothermal electric power"],
            "ssSMR": ["NG steam reforming - small-scale", "small-scale SMR"],
            "ssEl": ["electrolysis - small-scale", "small-scale electrolysis"],


        }
        mapping = {
                "ClGs": ["coal"],
                "POOl": ["oil"],
                "SMR": ["natural_gas"],
                "BioGs": ["biofuel"],
                "ClGsCS": ["coal", "CCS"],
                "POOlCS": ["oil", "CCS"],
                "SMRCS": ["natural_gas", "CCS"],
                "BioGsCS": ["biofuel", "CCS"],
                "BLEl": ["electrolysis"],
                "CurEl": ["electrolysis"],
                "SPVEl": ["electrolysis"],
                "CSPEl": ["electrolysis"],
                "WONEl": ["electrolysis"],
                "WOFFEl": ["electrolysis"],
                "GEOEl": ["electrolysis"],
                "ssSMR": ["natural_gas"],
                "ssEl": ["electrolysis"],
        }

        for source_type in ["coal", "oil", "natural_gas", "biofuel", "CCS", "electrolysis"]:
                h2_knowledge_graph.add(Node(name=source_type, inherits_from="HydrogenProduction"))

        for sub_type, synonyms in tech_types.items():
             h2_knowledge_graph.add(Node(name=sub_type, synonyms=synonyms, inherits_from=mapping[sub_type]))
    
        return h2_knowledge_graph

graph_h2 = create_h2_graph()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for t in h2cap["Type"].values:
    h2cap.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, color=COLOURS_HYDROGEN[t], label=str(t))

ax.legend()


In [ ]:
types = ["coal", "oil", "natural_gas", "biofuel", "electrolysis"]
test = graph_h2.aggregate_sum(h2cap, output_coords=types)

fig, ax = plt.subplots(figsize=(10, 6))
for t in test["Type"].values:
    test.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t))

ax.legend()